# LeetCode 1839 — Longest Substring Of All Vowels in Order bằng Python

Link đề: https://leetcode.com/problems/longest-substring-of-all-vowels-in-order/ · Độ khó: **Medium** · Chủ đề: String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def longestBeautifulSubstring(self, word: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa chuỗi **đẹp**: chứa **đủ 5** nguyên âm và các ký tự **không giảm** theo thứ tự `a < e < i < o < u`.
2. Nhận ra rằng trên bảng chữ chỉ gồm 5 nguyên âm, "đủ 5 nguyên âm + không giảm" ⇔ chuỗi **bắt đầu bằng `a`, kết thúc bằng `u`**, không giảm, và số **lần tăng** ký tự đúng bằng 4.
3. Giải bằng **một lượt quét** với hai biến: độ dài đoạn không giảm hiện tại và số loại nguyên âm trong đoạn.
4. Giải bằng **nén chạy** (run-length) rồi tìm 5 nhóm liên tiếp `a, e, i, o, u`; và bằng **biểu thức chính quy** `a+e+i+o+u+`.
5. Đối chiếu với vét cạn trên chuỗi ngẫu nhiên có seed cố định (nhiều trường hợp "gần đẹp").

## 2. Đề bài & ví dụ

Một chuỗi được gọi là **đẹp** nếu:
- Mỗi nguyên âm `'a', 'e', 'i', 'o', 'u'` xuất hiện **ít nhất một lần**.
- Các chữ cái được **sắp theo bảng chữ cái** (mọi `a` đứng trước mọi `e`, mọi `e` trước mọi `i`, …).

Ví dụ: `"aeiou"`, `"aaaaaaeiiiioou"` đẹp; `"uaeio"`, `"aeoiu"`, `"aaaeeeooo"` không đẹp.

Cho chuỗi `word` chỉ gồm nguyên âm, trả về độ dài **chuỗi con** (liên tiếp) đẹp **dài nhất**, hoặc `0` nếu không có.

**Ví dụ 1:** `word = "aeiaaioaaaaeiiiiouuuooaauuaeiu"` → `13` (`"aaaaeiiiiouuu"`).

**Ví dụ 2:** `word = "aeeeiiiioooauuuaeiou"` → `5` (`"aeiou"`).

**Ví dụ 3:** `word = "a"` → `0`.

**Ràng buộc:** `1 ≤ word.length ≤ 5·10⁵`; `word` chỉ gồm `'a','e','i','o','u'`.

In [1]:
import random
import time
import re
from itertools import groupby
from typing import List

examples = [("aeiaaioaaaaeiiiiouuuooaauuaeiu", 13), ("aeeeiiiioooauuuaeiou", 5), ("a", 0)]
for w, exp in examples:
    print(f"word={w!r} -> kỳ vọng {exp}")

word='aeiaaioaaaaeiiiiouuuooaauuaeiu' -> kỳ vọng 13
word='aeeeiiiioooauuuaeiou' -> kỳ vọng 5
word='a' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 So sánh ký tự

`'a' < 'e' < 'i' < 'o' < 'u'` theo mã Unicode — đúng thứ tự bảng chữ. Vậy "sắp theo bảng chữ cái" ⇔ `word[j] <= word[j+1]` với mọi `j`.

In [2]:
print([ (c, ord(c)) for c in "aeiou"], "| 'e' <= 'i':", "e" <= "i", "| 'o' <= 'e':", "o" <= "e")

[('a', 97), ('e', 101), ('i', 105), ('o', 111), ('u', 117)] | 'e' <= 'i': True | 'o' <= 'e': False


### 3.2 Kiểm tra một chuỗi có "đẹp" không

In [3]:
def is_beautiful(s):
    return set(s) == set("aeiou") and all(s[j] <= s[j + 1] for j in range(len(s) - 1))

for s in ["aeiou", "aaaaaaeiiiioou", "uaeio", "aeoiu", "aaaeeeooo"]:
    print(f"{s!r:<18} -> {is_beautiful(s)}")

'aeiou'            -> True
'aaaaaaeiiiioou'   -> True
'uaeio'            -> False
'aeoiu'            -> False
'aaaeeeooo'        -> False


### 3.3 `itertools.groupby` — nén chạy (run-length)

`groupby("aaeii")` sinh các nhóm ký tự liên tiếp giống nhau: `('a', 2), ('e', 1), ('i', 2)`.

In [4]:
print([(c, len(list(g))) for c, g in groupby("aaaaeiiiiouuu")])

[('a', 4), ('e', 1), ('i', 4), ('o', 1), ('u', 3)]


### 3.4 Biểu thức chính quy `re.finditer`

`a+e+i+o+u+` khớp "một hoặc nhiều `a`, rồi một hoặc nhiều `e`, …". `re.finditer` trả mọi đoạn khớp **không chồng nhau**, quét từ trái sang phải; với mẫu này các đoạn khớp chính là các đoạn đẹp tối đại.

In [5]:
w = examples[0][0]
print([(m.group(), m.start(), len(m.group())) for m in re.finditer(r"a+e+i+o+u+", w)])

[('aaaaeiiiiouuu', 7, 13)]


## 4. Ý tưởng nền tảng / trực giác

Chỉ có 5 ký tự, và chuỗi đẹp phải **không giảm** + đủ 5 loại ⇒ nó có dạng `a…a e…e i…i o…o u…u` với mỗi khối **không rỗng**. Nói cách khác:
- bắt đầu bằng `a`, kết thúc bằng `u`;
- trong đoạn, ký tự chỉ **giữ nguyên** hoặc **tăng lên đúng nguyên âm kế tiếp**; số lần tăng đúng 4.

Xét chuỗi theo các **đoạn không giảm tối đại** (cắt ở mỗi chỗ ký tự giảm). Hai nhận xét:
1. Mọi chuỗi con đẹp nằm trọn trong một đoạn như vậy (nó không giảm nên không vắt qua chỗ giảm).
2. Nếu một đoạn không giảm chứa một chuỗi con đẹp thì **cả đoạn** cũng đẹp: đoạn chứa đủ 5 nguyên âm; ký tự đầu của đoạn `≤ 'a'` nên phải là `'a'`, ký tự cuối `≥ 'u'` nên là `'u'`; và cả đoạn không giảm.

⇒ Đáp án = độ dài đoạn không giảm tối đại **dài nhất** trong số các đoạn chứa đủ 5 nguyên âm.

Ví dụ 1 cắt tại chỗ giảm:

| đoạn không giảm | đủ 5 & bắt đầu `a`? | độ dài |
|---|---|---|
| aei | không | 3 |
| aaio | không (thiếu e, u) | 4 |
| aaaaeiiiiouuu | **có** | **13** |
| oo | không | 2 |
| aauu | không | 4 |
| aeiu | không (thiếu o) | 4 |

In [6]:
def nondecreasing_runs(w):
    runs, start = [], 0
    for j in range(1, len(w) + 1):
        if j == len(w) or w[j] < w[j - 1]:
            runs.append(w[start:j]); start = j
    return runs

for seg in nondecreasing_runs(examples[0][0]):
    print(f"{seg!r:<16} đẹp? {is_beautiful(seg)}")

'aei'            đẹp? False
'aaio'           đẹp? False
'aaaaeiiiiouuu'  đẹp? True
'oo'             đẹp? False
'aauu'           đẹp? False
'aeiu'           đẹp? False


## 5. Cách 1 — Brute Force / baseline: kiểm tra mọi chuỗi con

In [7]:
def beautiful_bruteforce(word):
    n = len(word); best = 0
    for i in range(n):
        for j in range(i + 5, n + 1):                 # chuỗi đẹp dài ít nhất 5
            if is_beautiful(word[i:j]):
                best = max(best, j - i)
    return best

for w, exp in examples:
    got = beautiful_bruteforce(w); print(got, "kỳ vọng", exp); assert got == exp

13 kỳ vọng 13
5 kỳ vọng 5
0 kỳ vọng 0


**Độ phức tạp:** `O(n²)` chuỗi con × `O(n)` kiểm tra ⇒ `O(n³)` — với `n = 5·10⁵` là không thể.

### Cách 1' — Cố định điểm đầu `a`, mở rộng chừng nào còn không giảm

Với mỗi `i` có `word[i] == 'a'`, mở rộng `j` khi `word[j] >= word[j−1]`, đếm số loại; cập nhật đáp án khi đủ 5. `O(n²)` tệ nhất (chuỗi `"aaaa…a"`).

In [8]:
def beautiful_expand(word):
    n = len(word); best = 0
    for i in range(n):
        if word[i] != "a" or (i > 0 and word[i - 1] == "a"):   # chỉ bắt đầu ở 'a' đầu khối (tối ưu nhỏ)
            continue
        kinds = 1
        for j in range(i + 1, n):
            if word[j] < word[j - 1]:
                break
            if word[j] > word[j - 1]:
                kinds += 1
            if kinds == 5:
                best = max(best, j - i + 1)
    return best

print([beautiful_expand(w) for w, _ in examples])

[13, 5, 0]


**Nút thắt:** các lần mở rộng từ những điểm đầu khác nhau đi qua cùng một vùng nhiều lần. Nhưng như mục 4 chỉ ra, chỉ cần **một** lượt duyệt chia chuỗi thành các đoạn không giảm.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** duyệt một lượt, giữ:
- `length` = độ dài đoạn không giảm hiện tại (kết thúc tại vị trí `j`);
- `kinds` = số loại nguyên âm trong đoạn đó (= 1 + số lần ký tự **tăng**).

**Quy tắc tại mỗi `j ≥ 1`:**
- **Nếu** `word[j] < word[j−1]` (giảm) **thì** bắt đầu đoạn mới: `length = 1`, `kinds = 1`.
- **Nếu** `word[j] > word[j−1]` (tăng) **thì** `length += 1`, `kinds += 1`.
- **Nếu** bằng **thì** `length += 1`.
- **Nếu** `kinds == 5` **thì** `best = max(best, length)`.

**Chi tiết tinh tế:** đoạn phải **bắt đầu bằng `a`**. Nếu đoạn bắt đầu bằng ký tự khác (ví dụ `"eiou…"`), có thể có `kinds = 5`? Không — bắt đầu từ `e` thì tối đa 4 loại (`e, i, o, u`). Vì chỉ có 5 nguyên âm và đoạn không giảm, `kinds == 5` **tự động** kéo theo đoạn bắt đầu bằng `a` và đang ở `u`.

**Invariant:** sau bước `j`, `length` và `kinds` mô tả đúng đoạn không giảm dài nhất kết thúc tại `j`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Một lượt quét với `length` và `kinds`

In [9]:
def beautiful_scan(word):
    best, length, kinds = 0, 1, 1
    for j in range(1, len(word)):
        if word[j] < word[j - 1]:
            length, kinds = 1, 1
        else:
            if word[j] > word[j - 1]:
                kinds += 1
            length += 1
        if kinds == 5:
            best = max(best, length)
    return best

print([beautiful_scan(w) for w, _ in examples])

[13, 5, 0]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.2 Cách 3 — Nén chạy rồi tìm 5 nhóm liên tiếp `a, e, i, o, u`

**Giải thích:** nén `word` thành các cặp `(ký tự, độ dài)`. Một chuỗi đẹp tối đại ứng với **5 nhóm liên tiếp** có ký tự đúng `a, e, i, o, u`; độ dài = tổng 5 nhóm.

In [10]:
class SolutionGroups:
    def longestBeautifulSubstring(self, word: str) -> int:
        groups = [(c, len(list(g))) for c, g in groupby(word)]
        best = 0
        for i in range(len(groups) - 4):
            chars = "".join(c for c, _ in groups[i:i + 5])
            if chars == "aeiou":
                best = max(best, sum(L for _, L in groups[i:i + 5]))
        return best

print([SolutionGroups().longestBeautifulSubstring(w) for w, _ in examples])

[13, 5, 0]


**Vì sao đủ:** chuỗi đẹp dạng `a⁺e⁺i⁺o⁺u⁺` gồm **đúng 5 nhóm** chạy; muốn dài nhất thì lấy trọn mỗi nhóm (các nhóm chạy là tối đại). Hai nhóm kề nhau luôn khác ký tự nên không có trường hợp `a⁺` bị cắt đôi.

**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(số nhóm)` ≤ `O(n)`.

### 7.3 Cách 4 — Biểu thức chính quy `a+e+i+o+u+`

In [11]:
class SolutionRegex:
    def longestBeautifulSubstring(self, word: str) -> int:
        return max((len(m.group()) for m in re.finditer(r"a+e+i+o+u+", word)), default=0)

print([SolutionRegex().longestBeautifulSubstring(w) for w, _ in examples])

[13, 5, 0]


**Vì sao `finditer` không bỏ sót:** mẫu tham lam `a+` ăn **mọi** `a` liên tiếp ở vị trí bắt đầu; một khi khớp thất bại tại một vị trí, engine thử vị trí kế tiếp. Các chuỗi đẹp tối đại không chồng nhau (mỗi cái nằm trong một đoạn không giảm khác nhau) nên `finditer` tìm được tất cả. Có một tinh tế: nếu đoạn khớp bắt đầu ở giữa khối `a` thì sao? Không xảy ra — engine quét từ trái, gặp `a` đầu khối trước.

**Độ phức tạp Cách 4:** tuyến tính trong thực tế (mẫu không có backtracking phức tạp), cài bằng C nên rất nhanh.

### 7.4 Cách 5 — Máy trạng thái hữu hạn (DFA) theo "nguyên âm mong đợi"

Trạng thái = chỉ số nguyên âm hiện tại trong `"aeiou"` (hoặc `−1` = chưa bắt đầu). Ký tự giữ nguyên trạng thái ⇒ cộng độ dài; ký tự là nguyên âm **kế tiếp** ⇒ tiến trạng thái; còn lại ⇒ về trạng thái "chưa bắt đầu" (hoặc bắt đầu lại nếu ký tự là `a`). Đây là cách tư duy tổng quát hóa được cho chuỗi mẫu bất kỳ.

In [12]:
def beautiful_dfa(word):
    order = "aeiou"; pos = {c: i for i, c in enumerate(order)}
    state, length, best = -1, 0, 0
    for c in word:
        p = pos[c]
        if state >= 0 and (p == state or p == state + 1):
            state, length = p, length + 1
        elif p == 0:
            state, length = 0, 1                        # bắt đầu lại ở 'a'
        else:
            state, length = -1, 0
        if state == 4:
            best = max(best, length)
    return best

print([beautiful_dfa(w) for w, _ in examples])

[13, 5, 0]


**Độ phức tạp Cách 5:** `O(n)`, `O(1)`. Khác Cách 2 ở chỗ nó **chặn ngay** các bước nhảy hai bậc (ví dụ `a → i`). Với bảng chữ chỉ 5 nguyên âm thì Cách 2 vẫn đúng mà không cần chặn: một đoạn có bước nhảy hai bậc thiếu ít nhất một nguyên âm nên `kinds` không bao giờ đạt 5 (xem mục 10). Nhưng với mẫu tổng quát (Bài tập 5), DFA là cách an toàn.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** `"aeeeiiiioooauuuaeiou"`:

| j | c | so với trước | length | kinds | best |
|---|---|---|---|---|---|
| 0 | a | — | 1 | 1 | 0 |
| 1 | e | tăng | 2 | 2 | 0 |
| 2–3 | e | bằng | 4 | 2 | 0 |
| 4 | i | tăng | 5 | 3 | 0 |
| … | | | | | |
| 11 | a | **giảm** | 1 | 1 | 0 |
| 12 | u | tăng (a→u) | 2 | 2 | 0 |
| … | | | | | |
| 15 | a | giảm | 1 | 1 | 0 |
| 19 | u | tăng | 5 | 5 | **5** |

In [13]:
def trace_scan(word):
    best, length, kinds = 0, 1, 1
    print(f"j= 0 c='{word[0]}': length=1, kinds=1, best=0")
    for j in range(1, len(word)):
        c, p = word[j], word[j - 1]
        if c < p: length, kinds, rel = 1, 1, "giảm -> đoạn mới"
        elif c > p: kinds += 1; length += 1; rel = "tăng"
        else: length += 1; rel = "bằng"
        if kinds == 5: best = max(best, length)
        print(f"j={j:>2} c='{c}': {rel:<16} length={length:>2}, kinds={kinds}, best={best}")
    return best

assert trace_scan("aeeeiiiioooauuuaeiou") == 5

j= 0 c='a': length=1, kinds=1, best=0
j= 1 c='e': tăng             length= 2, kinds=2, best=0
j= 2 c='e': bằng             length= 3, kinds=2, best=0
j= 3 c='e': bằng             length= 4, kinds=2, best=0
j= 4 c='i': tăng             length= 5, kinds=3, best=0
j= 5 c='i': bằng             length= 6, kinds=3, best=0
j= 6 c='i': bằng             length= 7, kinds=3, best=0
j= 7 c='i': bằng             length= 8, kinds=3, best=0
j= 8 c='o': tăng             length= 9, kinds=4, best=0
j= 9 c='o': bằng             length=10, kinds=4, best=0
j=10 c='o': bằng             length=11, kinds=4, best=0
j=11 c='a': giảm -> đoạn mới length= 1, kinds=1, best=0
j=12 c='u': tăng             length= 2, kinds=2, best=0
j=13 c='u': bằng             length= 3, kinds=2, best=0
j=14 c='u': bằng             length= 4, kinds=2, best=0
j=15 c='a': giảm -> đoạn mới length= 1, kinds=1, best=0
j=16 c='e': tăng             length= 2, kinds=2, best=0
j=17 c='i': tăng             length= 3, kinds=3, best=0
j=18 c='o'

Lưu ý `j = 12`: từ `a` nhảy thẳng lên `u` vẫn tính là "tăng" ⇒ `kinds = 2`. Đoạn `"auuu"` chỉ 2 loại, không bao giờ đạt 5 trước khi bị cắt ở `j = 15` — vì vậy Cách 2 vẫn đúng dù không kiểm tra "tăng đúng một bậc".

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — một lượt quét.

In [14]:
class Solution:
    def longestBeautifulSubstring(self, word: str) -> int:
        best = 0
        length = kinds = 1
        for j in range(1, len(word)):
            if word[j] < word[j - 1]:
                length = kinds = 1
            else:
                if word[j] > word[j - 1]:
                    kinds += 1
                length += 1
            if kinds == 5:
                best = max(best, length)
        return best

for w, exp in examples:
    got = Solution().longestBeautifulSubstring(w); print(got, exp); assert got == exp

13 13
5 5
0 0


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Chuỗi con đẹp nằm trọn trong một đoạn không giảm tối đại (nó không giảm, nên không vắt qua chỗ giảm).

**Bổ đề 2.** Trong đoạn không giảm `D` (kết thúc tại `j`), `kinds` = số ký tự **khác nhau** trong `D` (mỗi lần tăng là gặp một ký tự mới lớn hơn, không bao giờ gặp lại ký tự cũ). Do đó `kinds = 5` ⇔ `D` chứa đủ 5 nguyên âm ⇔ `D` (không giảm, đủ 5) là chuỗi **đẹp**.

**Bổ đề 3 (tối ưu).** Nếu chuỗi con đẹp `S = word[i..j]` nằm trong đoạn không giảm `D` kết thúc tại `j`, thì `D` kết thúc tại `j` cũng đẹp (`D ⊇ S` vẫn không giảm và vẫn đủ 5 loại) và `|D| ≥ |S|`. Vậy với mỗi `j`, đoạn không giảm dài nhất kết thúc tại `j` là ứng viên tốt nhất kết thúc tại `j`. Thuật toán xét đúng các ứng viên này ⇒ đáp án là max của chúng. ∎

(Các bước "nhảy" như `a → i` không gây lỗi: `kinds` vẫn đếm đúng số loại khác nhau, chỉ là đoạn đó sẽ thiếu `e` nên không bao giờ đạt 5.)

### 10.1 Kiểm chứng thực nghiệm nhận xét "đoạn chứa chuỗi đẹp thì cả đoạn đẹp"

In [15]:
rng = random.Random(101)
checked = 0
for _ in range(2000):
    w = "".join(rng.choice("aeiou") for _ in range(rng.randint(5, 20)))
    for seg in nondecreasing_runs(w):
        has = any(is_beautiful(seg[i:j]) for i in range(len(seg)) for j in range(i + 5, len(seg) + 1))
        if has:
            assert is_beautiful(seg); checked += 1
print("Số đoạn không giảm có chứa chuỗi con đẹp đã kiểm:", checked, "- tất cả đều đẹp trọn vẹn")

Số đoạn không giảm có chứa chuỗi con đẹp đã kiểm: 11 - tất cả đều đẹp trọn vẹn


## 11. Phân tích Time / Space Complexity

Gọi `n = len(word) ≤ 5·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi chuỗi con | `O(n³)` | `O(n)` |
| 1'. Mở rộng từ mỗi `a` | `O(n²)` tệ nhất | `O(1)` |
| 2. Một lượt quét | `O(n)` | `O(1)` |
| 3. Nén chạy | `O(n)` | `O(n)` |
| 4. Regex | `O(n)` thực tế | `O(1)` + kết quả khớp |
| 5. DFA | `O(n)` | `O(1)` |

In [16]:
rng = random.Random(1839)
big = "".join(rng.choice("aeiou") for _ in range(5 * 10 ** 5))
long_ok = "a" * 100000 + "e" * 100000 + "i" * 100000 + "o" * 100000 + "u" * 100000
for name, f in [("scan", Solution().longestBeautifulSubstring), ("groups", SolutionGroups().longestBeautifulSubstring),
                ("regex", SolutionRegex().longestBeautifulSubstring), ("dfa", beautiful_dfa)]:
    for label, w in [("ngẫu nhiên", big), ("một khối dài", long_ok)]:
        t = time.perf_counter(); r = f(w); print(f"{name:<7} {label:<13}: {r:>6} ({(time.perf_counter()-t)*1000:.0f} ms)")

scan    ngẫu nhiên   :     11 (36 ms)


scan    một khối dài : 500000 (38 ms)


groups  ngẫu nhiên   :     11 (218 ms)
groups  một khối dài : 500000 (3 ms)
regex   ngẫu nhiên   :     11 (6 ms)
regex   một khối dài : 500000 (0 ms)
dfa     ngẫu nhiên   :     11 (31 ms)
dfa     một khối dài : 500000 (25 ms)


## 12. Bộ test

Chuỗi ngẫu nhiên thuần dễ không có chuỗi đẹp; ta trộn thêm chuỗi **có cấu trúc**: ghép các khối `a^x e^y i^z o^w u^v` (một số khối có thể rỗng hoặc đảo thứ tự) để tạo nhiều trường hợp "gần đẹp" (thiếu một nguyên âm, sai thứ tự một chỗ).

In [17]:
ALL = {"expand": beautiful_expand, "scan": beautiful_scan, "groups": SolutionGroups().longestBeautifulSubstring,
       "regex": SolutionRegex().longestBeautifulSubstring, "dfa": beautiful_dfa, "submit": Solution().longestBeautifulSubstring}
def run_all(w, with_brute=True):
    res = {k: f(w) for k, f in ALL.items()}
    if with_brute:
        res["brute"] = beautiful_bruteforce(w)
    assert len(set(res.values())) == 1, (w, res)
    return res["submit"]

for w, exp in examples:
    assert run_all(w) == exp
edge = [("aeiou", 5), ("uoiea", 0), ("aeio", 0), ("aaeiouu", 7), ("aeiouaeiou", 5), ("aeioua", 5), ("aeiouuuuu", 9), ("eiou", 0),
        ("aiou", 0), ("aeiuo", 0), ("aaaaa", 0)]
for w, exp in edge:
    assert run_all(w) == exp, w
rng = random.Random(18390)
def structured(rng):
    parts = []
    for _ in range(rng.randint(1, 4)):
        order = list("aeiou")
        if rng.random() < 0.3: rng.shuffle(order)
        for c in order:
            if rng.random() < 0.85: parts.append(c * rng.randint(1, 3))
    return "".join(parts)
for _ in range(400):
    run_all(structured(rng))
for _ in range(300):
    run_all("".join(rng.choice("aeiou") for _ in range(rng.randint(1, 25))))
print("Ví dụ, edge case, 400 chuỗi có cấu trúc, 300 chuỗi ngẫu nhiên: OK")

Ví dụ, edge case, 400 chuỗi có cấu trúc, 300 chuỗi ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra "không giảm", quên điều kiện đủ 5 nguyên âm

In [18]:
def wrong_no_count(word):
    best = length = 1
    for j in range(1, len(word)):
        length = 1 if word[j] < word[j - 1] else length + 1
        best = max(best, length)
    return best

w, exp = examples[2]
print("sai :", wrong_no_count(w), "| đúng:", exp)
print("sai :", wrong_no_count("aaaeeeooo"), "| đúng:", Solution().longestBeautifulSubstring("aaaeeeooo"))

sai : 1 | đúng: 0
sai : 9 | đúng: 0


**Vì sao sai:** `"aaaeeeooo"` không giảm nhưng thiếu `i`, `u`. **Sửa:** theo dõi `kinds` và chỉ cập nhật khi `kinds == 5`.

### Lỗi 2 — Quên reset `kinds` khi ký tự giảm

In [19]:
def wrong_no_reset_kinds(word):
    best, length, kinds = 0, 1, 1
    for j in range(1, len(word)):
        if word[j] < word[j - 1]:
            length = 1                                   # SAI: kinds không reset
        else:
            if word[j] > word[j - 1]: kinds += 1
            length += 1
        if kinds >= 5: best = max(best, length)
    return best

w = "aeiuaou"                                           # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_no_reset_kinds(w), "| đúng:", Solution().longestBeautifulSubstring(w), "| brute:", beautiful_bruteforce(w))

sai : 3 | đúng: 0 | brute: 0


**Vì sao sai:** số loại của đoạn cũ bị cộng dồn sang đoạn mới ⇒ `kinds` đạt 5 dù đoạn hiện tại chỉ có `a, o, u`. **Sửa:** `length = kinds = 1` khi giảm.

### Lỗi 3 — Dùng `set` của đoạn hiện tại nhưng không kiểm tra thứ tự (chấp nhận `"aeoiu"`)

In [20]:
def wrong_set_only(word):
    best = 0
    for i in range(len(word)):
        for j in range(i + 5, len(word) + 1):
            if set(word[i:j]) == set("aeiou"):              # SAI: thiếu điều kiện sắp xếp
                best = max(best, j - i)
    return best

w = "aeoiu"
print("sai :", wrong_set_only(w), "| đúng:", Solution().longestBeautifulSubstring(w))

sai : 5 | đúng: 0


**Vì sao sai:** `"aeoiu"` có đủ 5 nguyên âm nhưng `o` đứng trước `i`. **Sửa:** thêm điều kiện không giảm.

### Lỗi 4 — Bắt đầu `length = 0` (lệch một)

In [21]:
def wrong_off_by_one(word):
    best, length, kinds = 0, 0, 1                         # SAI: length đầu phải là 1 (ký tự word[0])
    for j in range(1, len(word)):
        if word[j] < word[j - 1]: length, kinds = 0, 1
        else:
            if word[j] > word[j - 1]: kinds += 1
            length += 1
        if kinds == 5: best = max(best, length)
    return best

w, exp = examples[0]
print("sai :", wrong_off_by_one(w), "| đúng:", exp)

sai : 12 | đúng: 13


**Vì sao sai:** đoạn bắt đầu tại `j` đã có 1 ký tự. **Sửa:** khởi tạo/reset `length = 1`.

### Lỗi 5 — Regex sai: `[aeiou]+` hoặc `a*e*i*o*u*`

In [22]:
def wrong_regex(word):
    return max((len(m.group()) for m in re.finditer(r"a*e*i*o*u*", word)), default=0)

w, exp = examples[2]
print("a*e*i*o*u* trên 'a':", wrong_regex(w), "| đúng:", exp)
print("trên 'aeio':", wrong_regex("aeio"), "| đúng:", Solution().longestBeautifulSubstring("aeio"))

a*e*i*o*u* trên 'a': 1 | đúng: 0
trên 'aeio': 4 | đúng: 0


**Vì sao sai:** `*` cho phép khối **rỗng** ⇒ khớp cả chuỗi thiếu nguyên âm. **Sửa:** `a+e+i+o+u+`.

### Lỗi 6 — Kiểm tra "nguyên âm kế tiếp" bằng `chr(ord(c) + 1)`

In [23]:
def wrong_next_letter(word):
    best, length, kinds = 0, 1, 1
    for j in range(1, len(word)):
        if word[j] == word[j - 1]:
            length += 1
        elif word[j] == chr(ord(word[j - 1]) + 1):        # SAI: 'e' không phải chữ ngay sau 'a'
            length += 1; kinds += 1
        else:
            length, kinds = 1, 1
        if kinds == 5: best = max(best, length)
    return best

w, exp = examples[0]
print("sai :", wrong_next_letter(w), "| đúng:", exp, "| chr(ord('a')+1) =", repr(chr(ord('a') + 1)))

sai : 0 | đúng: 13 | chr(ord('a')+1) = 'b'


**Vì sao sai:** các nguyên âm **không** liền nhau trong bảng chữ cái (`a` kế là `b`, không phải `e`). Nếu thật sự cần "nguyên âm kế tiếp", dùng chỉ số trong chuỗi `"aeiou"` (như DFA ở Cách 5). **Sửa:** so sánh `<`, `>` (Cách 2) hoặc tra `pos = {c: i for i, c in enumerate("aeiou")}`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi con dài nhất thỏa điều kiện cục bộ (không giảm)" ⇒ **một lượt quét**, reset khi vi phạm.
- Điều kiện "đủ các loại" + "không giảm" ⇒ đếm **số lần tăng** thay vì dùng `set`.
- Chuỗi trên bảng chữ nhỏ có cấu trúc khối ⇒ **run-length** (`groupby`).
- Mẫu cố định đơn giản ⇒ regex (`+` chứ không `*`).
- DFA theo "ký tự mong đợi kế tiếp" tổng quát hóa cho mẫu bất kỳ.

## 15. Biến thể của bài

1. **Không yêu cầu đủ 5 nguyên âm** (chỉ không giảm): bỏ điều kiện `kinds`.
2. **Mẫu bất kỳ** (ví dụ `"abc"` phải xuất hiện theo thứ tự, mỗi ký tự ≥ 1 lần): DFA tổng quát (Bài tập 5).
3. **Đếm số chuỗi con đẹp** (không phải dài nhất): với mỗi nhóm 5 khối liên tiếp `a^p e^q i^r o^s u^t` có `p · t` chuỗi con đẹp (chọn điểm đầu trong khối `a`, điểm cuối trong khối `u`).
4. **2062 Count Vowel Substrings of a String**: đếm chuỗi con chỉ gồm nguyên âm và đủ 5 loại (không cần thứ tự).
5. **674 Longest Continuous Increasing Subsequence**: phiên bản số của "một lượt quét, reset khi giảm".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con | `is_beautiful` | `O(n³)` | `O(n)` | Kiểm chứng |
| 1'. Mở rộng từ `a` | Dừng khi giảm | `O(n²)` | `O(1)` | Trung gian |
| 2. Một lượt quét | `length`, `kinds` | `O(n)` | `O(1)` | **Bản nộp** |
| 3. Nén chạy | 5 nhóm `aeiou` liên tiếp | `O(n)` | `O(n)` | Rõ cấu trúc, đếm chuỗi con |
| 4. Regex | `a+e+i+o+u+` | `O(n)` | `O(1)` | Code 1 dòng |
| 5. DFA | Trạng thái nguyên âm | `O(n)` | `O(1)` | Mẫu tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"aeiouu"`; (b) `"aaeeiioouuaeiou"`; (c) `"aeioaeiou"`; (d) `"uuuuu"`.

**Bài 2 (tính tay):** Liệt kê các đoạn không giảm tối đại của `"aeiouaaeeiioouuuie"` và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `count_increases(seg)` trả số lần ký tự tăng trong một chuỗi.

**Bài 4 (tự viết lại):** Viết `count_beautiful(word)` đếm **số chuỗi con đẹp** (theo vị trí) bằng nén chạy, `O(n)`.

**Bài 5 (mini challenge):** Viết `longest_pattern(word, pattern)`: độ dài chuỗi con dài nhất có dạng `p0⁺ p1⁺ … p_{m−1}⁺` với `pattern` gồm các ký tự **khác nhau** (thứ tự theo `pattern`, không nhất thiết theo bảng chữ cái). Đối chiếu vét cạn.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_increases_todo(seg):
    cnt = 0
    for j in range(1, len(seg)):
        # TODO: nếu seg[j] > seg[j-1] thì cnt += 1
        pass
    return cnt

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def count_beautiful(word):
    # TODO: nhóm chạy; mỗi 5 nhóm liên tiếp a,e,i,o,u đóng góp len(a-nhóm) * len(u-nhóm)
    pass

def longest_pattern(word, pattern):
    # TODO: DFA với vị trí trong pattern
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: cắt ở chỗ giảm. Mức 2: đếm loại. Mức 3: (a) `6`; (b) `10` (`"aaeeiioouu"`); (c) `5` (đoạn đầu `"aeio"` thiếu `u`); (d) `0`.

**Bài 2.** Mức 1: cắt tại các vị trí giảm (`u → a`, `u → i`, `i → e`). Mức 2: các đoạn: `"aeiou"` (5), `"aaeeiioouuu"` (11), `"i"`, `"e"`. Mức 3: hai đoạn đầu đủ 5 nguyên âm ⇒ đáp án `11`.

**Bài 3.** Mức 1: so sánh cặp kề. Mức 2: `>` nghiêm ngặt. Mức 3: chuỗi đẹp có đúng 4 lần tăng.

**Bài 4.** Mức 1: `groups = [(c, L), …]`. Mức 2: chuỗi con đẹp chọn điểm đầu trong khối `a` (bất kỳ trong `L_a` vị trí) và điểm cuối trong khối `u`. Mức 3: cộng `L_a · L_u` cho mỗi bộ 5 nhóm; đối chiếu vét cạn.

**Bài 5.** Mức 1: `pos = {c: i}`. Mức 2: giữ nguyên trạng thái hoặc tiến 1 ⇒ tiếp tục; gặp `pattern[0]` ⇒ bắt đầu lại; còn lại ⇒ reset. Mức 3: cập nhật khi trạng thái = `m − 1`; ký tự ngoài `pattern` luôn reset.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
b1 = [Solution().longestBeautifulSubstring(w) for w in ["aeiouu", "aaeeiioouuaeiou", "aeioaeiou", "uuuuu"]]
print("Bài 1:", b1); assert b1 == [6, 10, 5, 0]
w2 = "aeiouaaeeiioouuuie"
print("Bài 2:", nondecreasing_runs(w2), "->", Solution().longestBeautifulSubstring(w2)); assert Solution().longestBeautifulSubstring(w2) == 11
def count_increases_todo(seg):
    cnt = 0
    for j in range(1, len(seg)):
        if seg[j] > seg[j - 1]:
            cnt += 1
    return cnt
assert count_increases_todo("aaaaeiiiiouuu") == 4 and count_increases_todo("aaa") == 0
print("Bài 3: OK")

Bài 1: [6, 10, 5, 0]
Bài 2: ['aeiou', 'aaeeiioouuu', 'i', 'e'] -> 11
Bài 3: OK


In [27]:
# Bài 4, 5
def count_beautiful(word):
    g = [(c, len(list(it))) for c, it in groupby(word)]
    return sum(g[i][1] * g[i + 4][1] for i in range(len(g) - 4) if "".join(c for c, _ in g[i:i + 5]) == "aeiou")

def longest_pattern(word, pattern):
    pos = {c: i for i, c in enumerate(pattern)}; m = len(pattern)
    state, length, best = -1, 0, 0
    for c in word:
        p = pos.get(c, -2)
        if state >= 0 and (p == state or p == state + 1):
            state, length = p, length + 1
        elif p == 0:
            state, length = 0, 1
        else:
            state, length = -1, 0
        if state == m - 1:
            best = max(best, length)
    return best

def pattern_brute(word, pattern):
    rx = re.compile("".join(re.escape(c) + "+" for c in pattern))
    return max((j - i for i in range(len(word)) for j in range(i + 1, len(word) + 1) if rx.fullmatch(word[i:j])), default=0)

rng = random.Random(6)
for _ in range(300):
    w = structured(rng)
    assert count_beautiful(w) == sum(1 for i in range(len(w)) for j in range(i + 5, len(w) + 1) if is_beautiful(w[i:j]))
    pat = "".join(rng.sample("abcd", rng.randint(1, 4)))
    ww = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 14)))
    assert longest_pattern(ww, pat) == pattern_brute(ww, pat), (ww, pat)
    assert longest_pattern(w, "aeiou") == Solution().longestBeautifulSubstring(w)
print("Bài 4, 5: OK |", count_beautiful("aaeiouu"), longest_pattern("ccbbbaacba", "cba"))

Bài 4, 5: OK | 4 7


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "một lượt quét, reset khi giảm, đếm số lần tăng":**

```python
best, length, kinds = 0, 1, 1
for j in range(1, len(word)):
    if word[j] < word[j-1]: length = kinds = 1
    else:
        if word[j] > word[j-1]: kinds += 1
        length += 1
    if kinds == 5: best = max(best, length)
return best
```

**Checklist:**
- [ ] Kiểm tra **cả** không giảm **và** đủ 5 nguyên âm?
- [ ] Reset **cả** `length` và `kinds` khi giảm?
- [ ] `length` bắt đầu từ 1?
- [ ] Regex dùng `+` (không `*`)?
- [ ] Đã test: chuỗi ngắn hơn 5, thiếu một nguyên âm, sai thứ tự một chỗ, khối dài `5·10⁵`?